# KANQAS-NISQ Quickstart

Hardware-aware Curriculum RL Quantum Architecture Search using KAN policy networks.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer
from rich.console import Console

console = Console()
console.print("[bold green]KANQAS-NISQ ready![/bold green]")

## 1. Generate Molecular Hamiltonians

In [ ]:
from chemistry.molecule import MolecularHamiltonian

h2 = MolecularHamiltonian.h2(bond_length=0.74)
console.print(f"H2: {h2.num_qubits} qubits, ref E = {h2.reference_energy:.6f} Ha")
console.print(f"Exact GS: {h2.exact_diagonalization():.6f} Ha")

lih = MolecularHamiltonian.lih(bond_length=2.0)
console.print(f"LiH: {lih.num_qubits} qubits, ref E = {lih.reference_energy:.6f} Ha")

## 2. Create a Quantum Circuit

In [ ]:
circ = QuantumCircuit(4)
circ.h(0)
circ.cx(0, 1)
circ.cx(0, 2)
circ.cx(0, 3)
circuit_drawer(circ, output='mpl')

## 3. Evaluate Energy

In [ ]:
energy = h2.estimate_energy(circ)
exact = h2.exact_diagonalization()
console.print(f"Circuit energy: {energy:.6f} Ha")
console.print(f"Exact GS: {exact:.6f} Ha")
console.print(f"Error: {abs(energy - exact):.6f} Ha")

## 4. Run H2 VQE Training

In [ ]:
from chemistry.h2_vqe import H2VQETrainer

trainer = H2VQETrainer(bond_lengths=[0.74], agent_type='KAQN')
results = trainer.run()
console.print(results)

## 5. Visualize Results

In [ ]:
from interpretability.kan_visualizer import KANVisualizer

viz = KANVisualizer()
if 'bond_lengths' in results:
    viz.plot_energy_curve(
        results['bond_lengths'],
        results['found_energies'],
        results['exact_energies'],
        molecule_name='H2'
    )

## 6. Launch Dashboard

In [ ]:
# Uncomment to run:
# !streamlit run ../interpretability/streamlit_dashboard.py